# 실습 3. 결정 트리로 피싱사이트 분류 + 세 모델 비교

**과목:** 인공지능과 정보보호 | **과제 3·4번**  
**목표:** 피싱 URL 데이터셋에서 결정 트리를 학습하고 트리를 시각화한다. 깊이별 훈련/테스트 점수를 비교한다. 마지막으로 같은 피싱 데이터에서 **KNN·로지스틱 회귀·결정 트리**를 동일 조건으로 5-폴드 교차 검증해 비교한다.

> 주의: 실습 1의 KNN은 악성코드 데이터이고, 실습 2·3의 모델은 피싱 데이터이므로 원래 점수끼리 단순 비교하면 공정하지 않다. 여기서는 세 알고리즘을 **같은 피싱 데이터**에 다시 학습해 공정하게 비교한다.

데이터 출처: https://www.kaggle.com/datasets/shashwatwork/web-page-phishing-detection-dataset

> 실행 환경: Google Colab (Python 3). 외부 CSV를 원본 데이터셋에서 다운로드하고 학습·평가를 실행한다.

In [ ]:
# 공개 Kaggle 데이터셋 다운로드를 위한 도구를 코랩에 설치한다.
%pip -q install kagglehub

In [ ]:
# 파일 경로 관리와 표 데이터를 다루기 위한 라이브러리를 불러온다.
from pathlib import Path
import pandas as pd
import numpy as np
import kagglehub

# 강의자료가 지정한 피싱 URL 데이터셋과 CSV 파일명을 사용한다.
DATASET_ID = 'shashwatwork/web-page-phishing-detection-dataset'
EXPECTED_CSV = 'dataset_phishing.csv'
try:
    data_folder = Path(kagglehub.dataset_download(DATASET_ID))
    csv_files = list(data_folder.rglob('*.csv'))
    matched = [p for p in csv_files if p.name.lower() == EXPECTED_CSV.lower()]
    csv_path = matched[0] if matched else (csv_files[0] if len(csv_files)==1 else None)
    if csv_path is None:
        raise FileNotFoundError(f'원하는 CSV 파일을 찾을 수 없습니다: {csv_files}')
except Exception as err:
    # Kaggle 자동 다운로드가 실패하면 원본 CSV 업로드로 이어갈 수 있게 한다.
    print('자동 다운로드 실패:', type(err).__name__, str(err)[:220])
    print('Kaggle에서 dataset_phishing.csv를 받아 업로드하세요.')
    from google.colab import files
    uploaded = files.upload()
    csv_path = Path(next((n for n in uploaded if n.lower().endswith('.csv')), ''))
    if not csv_path.is_file():
        raise FileNotFoundError('dataset_phishing.csv 업로드가 필요합니다.')

# 원본 URL 데이터셋과 정답 분포를 확인한다.
url = pd.read_csv(csv_path)
print('데이터 파일:', csv_path.name)
print('원본 크기(행, 열):', url.shape)
print('원본 정답 수:', url['status'].value_counts().to_dict())
print('열 이름 일부:', list(url.columns[:12]))

## 1. 데이터 전처리와 독립적인 80:20 분할

실습 2와 동일한 규칙을 사용한다. 결정 트리는 표준화가 필수는 아니지만 결측 보정을 위해 `SimpleImputer`를 모델 파이프라인에 넣었다.

In [ ]:
# legitimate=0(정상), phishing=1(피싱)으로 레이블을 통일한다.
labels = url['status'].astype(str).str.strip().str.lower()
target = labels.map({'legitimate': 0, 'phishing': 1})
if target.isna().any():
    raise ValueError(f'알 수 없는 status 값: {sorted(labels[target.isna()].unique())}')
target = target.astype(int)

# 원본 URL 문자 열과 정답 열을 학습 입력에서 제외한다.
# 수업에서는 이미 추출된 숫자형 URL 피처를 학습에 사용한다.
data = url.drop(columns=['url', 'status'], errors='ignore')
# 숫자처럼 저장된 문자열도 숫자형으로 바꾸고, 변환 불가능한 값은 NaN으로 표시한다.
data = data.apply(pd.to_numeric, errors='coerce')
# 숫자형 값이 하나도 없는 열은 학습할 정보가 없어 제외한다.
empty_cols = data.columns[data.isna().all()].tolist()
if empty_cols:
    print('전체 결측 열 제외:', empty_cols)
    data = data.drop(columns=empty_cols)

print('입력 피처 크기:', data.shape, '/ 레이블:', target.value_counts().to_dict())
print('결측값 총 개수:', int(data.isna().sum().sum()))
assert data.shape[1] > 0, '학습에 사용할 숫자 피처가 없습니다.'

# 수업과 같이 80%는 훈련, 20%는 테스트에 사용한다.
# 층화 분할(stratify)로 각 클래스의 비율을 비슷하게 유지한다.
from sklearn.model_selection import train_test_split
train_input, test_input, train_target, test_target = train_test_split(
    data, target, test_size=0.2, random_state=42, stratify=target
)
print('훈련:', train_input.shape, '테스트:', test_input.shape)

## 2. 깊이 3 결정 트리 학습 및 성능

In [ ]:
# 결정 트리 모델과 시각화 함수를 준비한다.
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt

# 깊이를 3으로 제한한 트리 모델을 만든다.
dt3 = make_pipeline(
    SimpleImputer(strategy='median'),
    DecisionTreeClassifier(max_depth=3, random_state=42)
)
# 훈련 데이터로 질문 규칙(분할 기준)을 학습한다.
dt3.fit(train_input, train_target)
# 훈련 정확도와 테스트 정확도를 각각 출력한다.
print('훈련 정확도:', f'{dt3.score(train_input,train_target):.4f}')
print('테스트 정확도:', f'{dt3.score(test_input,test_target):.4f}')
# 클래스 0은 정상, 1은 피싱임을 유지해 혼동행렬을 확인한다.
dt_pred = dt3.predict(test_input)
print('혼동행렬:\n', confusion_matrix(test_target,dt_pred,labels=[0,1]))
print(classification_report(test_target,dt_pred,target_names=['legitimate','phishing'],zero_division=0))

## 3. 결정 트리 실제 그림 출력·해석

- 왼쪽 가지: `특성 ≤ 기준값`이 **참(True)**
- 오른쪽 가지: 조건이 **거짓(False)**
- `gini`: 클래스가 섞여 있는 정도 (0이면 한 클래스만 존재)
- `samples`: 해당 노드에 들어온 훈련 샘플 수
- `value`: `[정상 수, 피싱 수]`
- `class`: 그 노드에서 가장 많은 클래스

그림과 아래 텍스트에서 **루트 노드와 다음 두 단계 조건**을 직접 확인한다.

In [ ]:
# 파이프라인에 들어 있는 실제 결정 트리 객체를 가져온다.
tree_model = dt3.named_steps['decisiontreeclassifier']
# 피처 이름과 정상/피싱 이름을 트리에 표시한다.
plt.figure(figsize=(22,10))
plot_tree(tree_model, feature_names=list(data.columns),
          class_names=['legitimate','phishing'], filled=True,
          rounded=True, fontsize=8)
plt.title('Phishing decision tree (max_depth=3)')
plt.tight_layout()
# 그림 파일도 코랩 작업 공간에 저장한다.
plt.savefig('phishing_decision_tree_depth3.png',dpi=160,bbox_inches='tight')
plt.show()

# 모든 분기 질문을 글로 읽어볼 수 있게 트리 규칙을 출력한다.
print(export_text(tree_model,feature_names=list(data.columns),max_depth=3))
# 루트 노드에 쓰인 특성을 출력한다.
root_idx = int(tree_model.tree_.feature[0])
print('루트(첫 질문)의 특성:', data.columns[root_idx])

## 4. 결정 트리 깊이 바꿔가며 과대적합 확인

In [ ]:
# 강의자료의 깊이 후보를 그대로 반복 비교한다.
depth_results = []
for depth in [1, 3, 5, 10, None]:
    model = make_pipeline(SimpleImputer(strategy='median'),
                          DecisionTreeClassifier(max_depth=depth,random_state=42))
    model.fit(train_input,train_target)
    depth_results.append({'max_depth':str(depth),
                          '훈련 정확도':model.score(train_input,train_target),
                          '테스트 정확도':model.score(test_input,test_target)})
# 깊이에 따라 훈련 점수와 테스트 점수 차이가 커지는지 살펴본다.
depth_df = pd.DataFrame(depth_results)
display(depth_df.round(4))
depth_df.set_index('max_depth')[['훈련 정확도','테스트 정확도']].plot(marker='o',figsize=(7,4))
plt.ylabel('Accuracy')
plt.title('Tree depth vs accuracy')
plt.grid(alpha=0.3)
plt.show()
print('주의: 깊이 선택은 별도 검증 세트나 교차 검증으로 해야 테스트 평가가 공정하다.')

## 5. 중요 피처 출력

`feature_importances_`는 트리의 불순도 감소에 대한 상대적 기여도를 나타낸다. 모델의 첫 분할 피처가 반드시 전체 중요도 1위와 같지는 않을 수 있다. 아래 셀로 직접 확인한다.

In [ ]:
# 깊이 3 트리가 계산한 피처 중요도를 표로 정리한다.
importance = pd.Series(tree_model.feature_importances_,index=data.columns)
print('[결정 트리 피처 중요도 상위 10개]')
display(importance.sort_values(ascending=False).head(10).to_frame('importance'))
print('트리의 첫 질문 특성:', data.columns[int(tree_model.tree_.feature[0])])

## 6. 같은 피싱 데이터에서 세 모델 비교

**비교의 공정성:** 세 모델은 같은 URL 데이터, 같은 정답 변환, 동일한 훈련/테스트 분할을 사용한다. KNN과 로지스틱 회귀에만 표준화를 적용하고, 결정 트리는 숫자값을 그대로 사용한다. 모든 모델은 **Stratified 5-Fold**를 사용해 교차 검증한다.

실습 1 악성코드 KNN 결과는 여기 표에 섞지 않는다.

In [ ]:
# 공정한 모델 비교를 위해 세 분류기를 정의한다.
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import precision_score, recall_score, f1_score
from time import perf_counter

models = {
    'KNN (k=5)': make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),
                               KNeighborsClassifier(n_neighbors=5,n_jobs=-1)),
    'Logistic Regression': make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),
                                         LogisticRegression(max_iter=2000,random_state=42)),
    'Decision Tree (depth=3)': make_pipeline(SimpleImputer(strategy='median'),
                                          DecisionTreeClassifier(max_depth=3,random_state=42)),
}
# 같은 훈련 데이터에 대해 5-폴드 교차 검증을 수행한다.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
records = []
for name, model in models.items():
    # 각 검증 폴드의 accuracy, precision, recall, F1과 소요 시간을 측정한다.
    cv_result = cross_validate(
        model,train_input,train_target,cv=cv,
        scoring={'accuracy':'accuracy','precision':'precision','recall':'recall','f1':'f1'},
        n_jobs=1, return_train_score=False
    )
    # 교차 검증 후 전체 훈련 세트로 최종 모델을 학습하고 시간을 잰다.
    start=perf_counter()
    model.fit(train_input,train_target)
    fit_seconds=perf_counter()-start
    test_pred=model.predict(test_input)
    # 테스트 데이터는 최종 점수에만 사용한다.
    records.append({
        '모델':name,
        'CV 정확도 평균':cv_result['test_accuracy'].mean(),
        'CV 정확도 표준편차':cv_result['test_accuracy'].std(),
        'CV F1 평균':cv_result['test_f1'].mean(),
        '테스트 정확도':accuracy_score(test_target,test_pred),
        '테스트 정밀도':precision_score(test_target,test_pred,zero_division=0),
        '테스트 재현율':recall_score(test_target,test_pred,zero_division=0),
        '테스트 F1':f1_score(test_target,test_pred,zero_division=0),
        '최종 fit 시간(초)':fit_seconds,
    })
comparison_df=pd.DataFrame(records)
display(comparison_df.round(4))

## 7. 모델 비교 시각화 및 결론

In [ ]:
# 같은 테스트 데이터를 사용한 세 모델의 성능을 막대그래프로 표시한다.
comparison_df.set_index('모델')[['테스트 정확도','테스트 재현율','테스트 F1']].plot(
    kind='bar',rot=0,figsize=(10,5)
)
plt.ylim(0,1)
plt.ylabel('Score')
plt.title('Three models on the SAME phishing test set')
plt.xticks(rotation=12,ha='right')
plt.tight_layout()
plt.show()

# 검증 점수와 실제 피싱 탐지 재현율을 구분해 비교한다.
cv_best=comparison_df.loc[comparison_df['CV 정확도 평균'].idxmax()]
recall_best=comparison_df.loc[comparison_df['테스트 재현율'].idxmax()]
print(f"1. 5-폴드 검증 정확도 평균은 {cv_best['모델']} 모델이 {cv_best['CV 정확도 평균']:.2%}로 가장 높았다.")
print(f"2. 최종 테스트에서 피싱 재현율은 {recall_best['모델']} 모델이 {recall_best['테스트 재현율']:.2%}로 가장 높았다.")
print('3. 보안 문제는 전체 정확도뿐 아니라 미탐 위험과 판단 근거, 학습·예측 비용을 함께 고려해야 한다.')

## 8. 최종 정리 및 직접 확인

| 모델 | 분류 방법 | 해석 가능성 |
|:--|:--|:--|
| KNN | 가까운 데이터 K개의 다수결 | 비교적 낮음 (이웃 확인 필요) |
| 로지스틱 회귀 | 피처 계수를 더해 피싱 확률 계산 | 보통 (계수 해석 가능) |
| 결정 트리 | 예/아니오 조건을 차례로 확인 | 높음 (트리 그림·경로 확인 가능) |

- `cv=5`는 매번 훈련 데이터의 4/5로 학습하고 1/5로 검증하는 것을 5번 반복한다는 뜻이다.
- 편차(표준편차)가 크다면 분할에 따른 성능 변화가 크다는 뜻이며, 분할 편향 등 원인을 추가 확인해야 한다.
- 실습 1과 실습 2·3은 데이터가 다르다. 따라서 동일 데이터에서 비교한 **위의 세 모델 표**를 4번 과제의 근거로 사용한다.
- 실제 트리의 질문을 보안 관점에서 분석하고 결과가 타당한지 검토한다.